# Model Instantiation Test

This notebook only constructs local model instances, including the 8-layer Hydra/Transformer hybrid. It does not train or load checkpoints.

In [1]:
import sys
from pathlib import Path

repo_root = Path("/app")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import tabpfn.encoders as encoders

from tabpfn.hydra import HydraModel
from tabpfn.hybrid import HybridHydraTransformerModel
from tabpfn.mamba import MambaModel
from tabpfn.transformer import TransformerModel


/usr/local/lib/python3.8/dist-packages/mamba_ssm/ops/selective_scan_interface.py:164: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, xz, conv1d_weight, conv1d_bias, x_proj_weight, delta_proj_weight,
/usr/local/lib/python3.8/dist-packages/mamba_ssm/ops/selective_scan_interface.py:240: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, dout):
/usr/local/lib/python3.8/dist-packages/mamba_ssm/ops/triton/layer_norm.py:986: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(
/usr/local/lib/python3.8/dist-packages/mamba_ssm/ops/triton/layer_norm.py:1045: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cud

In [6]:
assert torch.cuda.is_available(), (
    "Hydra requires CUDA; this notebook kernel has no GPU access."
)

device = torch.device("cuda:4")
torch.cuda.set_device(device)

In [7]:
#device = "cuda:7" if torch.cuda.is_available() else "cpu"

num_features = 10
num_classes = 3
emsize = 64
nhead = 4
nhid = 128
nlayers = 2

hydra_x_encoder = encoders.Linear(num_features, emsize)
hydra_y_encoder = encoders.Linear(1, emsize)

hybrid_8l_x_encoder = encoders.Linear(num_features, emsize)
hybrid_8l_y_encoder = encoders.Linear(1, emsize)

mamba_x_encoder = encoders.Linear(num_features, emsize)
mamba_y_encoder = encoders.Linear(1, emsize)

trans_x_encoder = encoders.Linear(num_features, emsize)
trans_y_encoder = encoders.Linear(1, emsize)


In [8]:
hydra_test = HydraModel(
    encoder=hydra_x_encoder,
    n_out=num_classes,
    ninp=emsize,
    nhid=nhid,
    y_encoder=hydra_y_encoder,
    num_layers=nlayers,
    device=device,
).to(device)

hybrid_8l = HybridHydraTransformerModel(
    encoder=hybrid_8l_x_encoder,
    n_out=num_classes,
    ninp=emsize,
    nhead=nhead,
    nhid=nhid,
    nlayers=8,
    y_encoder=hybrid_8l_y_encoder,
    efficient_eval_masking=True,
    device=device,
).to(device)

mamba_test = MambaModel(
    encoder=mamba_x_encoder,
    n_out=num_classes,
    ninp=emsize,
    nhid=nhid,
    y_encoder=mamba_y_encoder,
    num_layers=nlayers,
    device=device,
).to(device)

trans_test = TransformerModel(
    encoder=trans_x_encoder,
    n_out=num_classes,
    ninp=emsize,
    nhead=nhead,
    nhid=nhid,
    nlayers=nlayers,
    y_encoder=trans_y_encoder,
    efficient_eval_masking=True,
).to(device)

hydra_test.eval()
hybrid_8l.eval()
mamba_test.eval()
trans_test.eval()

print(f"hydra_test params: {sum(p.numel() for p in hydra_test.parameters()):,}")
print(f"hybrid_8l params: {sum(p.numel() for p in hybrid_8l.parameters()):,}")
print(f"mamba_test params: {sum(p.numel() for p in mamba_test.parameters()):,}")
print(f"trans_test params: {sum(p.numel() for p in trans_test.parameters()):,}")


hydra_test params: 99,407
hybrid_8l params: 323,163
mamba_test params: 75,331
trans_test params: 76,483


In [9]:
# Optional smoke test: run one tiny forward pass through all models.
seq_len = 16
batch_size = 2
single_eval_pos = 8

x = torch.randn(seq_len, batch_size, num_features, device=device)
y = torch.randint(0, num_classes, (seq_len, batch_size), device=device).float()

with torch.no_grad():
    hydra_out = hydra_test((x, y), single_eval_pos=single_eval_pos)
    hybrid_8l_out = hybrid_8l((x, y), single_eval_pos=single_eval_pos)
    mamba_out = mamba_test((x, y), single_eval_pos=single_eval_pos)
    trans_out = trans_test((x, y), single_eval_pos=single_eval_pos)

print("hydra_out shape:", tuple(hydra_out.shape))
print("hybrid_8l_out shape:", tuple(hybrid_8l_out.shape))
print("mamba_out shape:", tuple(mamba_out.shape))
print("trans_out shape:", tuple(trans_out.shape))


hydra_out shape: (8, 2, 3)
hybrid_8l_out shape: (8, 2, 3)
mamba_out shape: (8, 2, 3)
trans_out shape: (8, 2, 3)


In [10]:
print(hybrid_8l)

HybridHydraTransformerModel(
  (encoder): Linear(in_features=10, out_features=64, bias=True)
  (y_encoder): Linear(in_features=1, out_features=64, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=3, bias=True)
  )
  (hybrid_encoder): HybridEncoder(
    (layers): ModuleList(
      (0): HydraEncoderLayer(
        (block): HydraBlock(
          (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (mixer): Hydra(
            (in_proj): Linear(in_features=64, out_features=516, bias=False)
            (conv1d): Conv1d(384, 384, kernel_size=(7,), stride=(1,), padding=(3,), groups=384)
            (act): SiLU()
            (fc_D): Linear(in_features=128, out_features=2, bias=False)
            (norm): RMSNorm()
            (out_proj): Linear(in_features=128, out_features=64, bias=False)
          )
        )
        (norm_f): LayerNorm((64,), eps=1e-